# 26. 근접 중복쌍 매칭을 이용한 stress_score 예측

train과 test에 같은 원본 레코드를 두 번 넣은 흔적이 있다. 숫자 컬럼에만 미세한 값 차이가
있고 범주형 7개와 `stress_score`는 쌍 안에서 동일하다. 이 구조를 찾아 test 행마다
가장 가까운 train 행의 정답을 가져오는 방식(k=1 최근접 이웃)으로 예측한다.

같이 확인하는 것:

- 기존 CV 점수(LGBM 0.18, SVR 0.15)가 이 중복쌍 암기에서 나온 것인지
- 파생변수 19개가 누수 없는 검증에서도 효과가 있는지
- 대회 누수 규정을 위반하지 않는지


## 1. 설정

In [2]:
import numpy as np
import pandas as pd
import warnings
import lightgbm as lgb
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import mean_absolute_error as mae

warnings.filterwarnings('ignore')

CAT = ['gender', 'activity', 'smoke_status', 'medical_history',
       'family_medical_history', 'sleep_pattern', 'edu_level']
NUM = ['age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure',
       'diastolic_blood_pressure', 'glucose', 'bone_density']

TH = 0.45          # 쌍으로 인정할 거리 상한 (3절에서 선택 근거)
FALLBACK = 0.50    # 매칭 실패 시 기본값 (균등분포의 MAE 최적 상수)
OVERWORK_TH = 11   # mean_working 이 이 값 이상이면 폴백을 따로 준다 (6절에서 선택 근거)

tr = pd.read_csv('../data/train.csv')
te = pd.read_csv('../data/test.csv')
y = tr.stress_score.values
print('train', tr.shape, ' test', te.shape)

train (3000, 18)  test (3000, 17)


## 2. 피처와 타겟의 관계 확인

모델링 방향을 정하기 전에 피처에 예측 가능한 신호가 있는지 본다.
숫자형은 상관계수로, 범주형은 그룹별 타겟 평균이 전체 평균에서 얼마나 벗어나는지로 확인한다.

In [4]:
print('[타겟 분포]')
print(f'  평균 {y.mean():.4f}   표준편차 {y.std():.4f}'
      f'   (균등분포 U[0,1]의 표준편차 = {1 / np.sqrt(12):.4f})')
print(f'  값 종류 {pd.Series(y).nunique()}개, 범위 {y.min()}~{y.max()}, '
      f'값당 평균 {len(y) / pd.Series(y).nunique():.1f}행')

print()
print('[숫자 피처와 타겟의 상관]')
for c in NUM + ['mean_working']:
    m = tr[c].notna()
    print(f'  {c:28s} r = {tr.loc[m, c].corr(pd.Series(y)[m]):+.4f}')

print()
worst = max(abs(tr.groupby(tr[c].fillna('__NA__')).stress_score.mean() - y.mean()).max()
            for c in CAT)
print(f'[범주형] 그룹별 타겟 평균이 전체 평균에서 벗어나는 최대폭: {worst:.4f}')

[타겟 분포]
  평균 0.4821   표준편차 0.2882   (균등분포 U[0,1]의 표준편차 = 0.2887)
  값 종류 101개, 범위 0.0~1.0, 값당 평균 29.7행

[숫자 피처와 타겟의 상관]
  age                          r = +0.0187
  height                       r = -0.0057
  weight                       r = +0.0113
  cholesterol                  r = +0.0213
  systolic_blood_pressure      r = +0.0156
  diastolic_blood_pressure     r = +0.0254
  glucose                      r = -0.0061
  bone_density                 r = -0.0226
  mean_working                 r = +0.1834

[범주형] 그룹별 타겟 평균이 전체 평균에서 벗어나는 최대폭: 0.0291


타겟은 0.00~1.00을 0.01 간격으로 채운 균등분포이고, 모든 피처의 상관이 |r| < 0.03이다.
개별 피처로는 예측할 수 있는 게 없다.

## 3. 기본 CV 결과 확인

신호가 없는데 기존 노트북들의 CV는 상수 예측(0.25)보다 좋게 나왔다. 먼저 재현한다.

In [6]:
X = tr.drop(columns=['ID', 'stress_score']).copy()
for c in X.select_dtypes('object'):
    X[c] = X[c].astype('category')

oof = np.zeros(len(y))
for t, v in KFold(5, shuffle=True, random_state=42).split(X):
    m = lgb.LGBMRegressor(n_estimators=1500, learning_rate=0.03, verbose=-1)
    m.fit(X.iloc[t], y[t], eval_set=[(X.iloc[v], y[v])], eval_metric='l1',
          callbacks=[lgb.early_stopping(100, verbose=False)])
    oof[v] = m.predict(X.iloc[v])

print(f'상수(중앙값) MAE : {mae(y, np.full(len(y), np.median(y))):.4f}')
print(f'LGBM 5-Fold  MAE : {mae(y, oof):.4f}')

상수(중앙값) MAE : 0.2494
LGBM 5-Fold  MAE : 0.1846


## 4. 근접 중복쌍 탐색

데이터를 훑어보면 거의 같은 행이 두 개씩 있다. 이걸 찾는 방법:

1. **블로킹** — 범주형 7개가 완전히 일치하는 행끼리만 후보로 묶는다.
   3000행 전체를 비교하면 O(n²)이지만, 블록으로 나누면 블록 하나가 수십 행이라 충분히 빠르다.
2. **체비셰프 거리** — 숫자 8개를 각각 표준편차로 나눈 뒤, 가장 크게 어긋난 컬럼 하나의
   크기를 거리로 쓴다. 지터가 모든 컬럼에 골고루 작게 들어가 있어서, 차이를 합산하는
   유클리드보다 "최대 차이"를 보는 체비셰프가 쌍을 더 깔끔하게 갈라낸다.
3. 거리가 임계값 이하인 행끼리 연결하고 연결 요소로 묶는다.

이 절은 train 3000행만 쓴다. 거리 척도도 train 표준편차다 (규정 관련, 5절 표 참고).

In [8]:
def pair_labels(df, scale, th=TH):
    """가까운 행끼리 묶어서 그룹 번호를 반환.

    scale 은 호출하는 쪽이 train 통계로 넘긴다.
    함수 안에서 입력 df 의 표준편차를 구하지 않는다.
    """
    sig = df[CAT].fillna('__NA__').agg('|'.join, axis=1).values
    V = df[NUM].values.astype(float)

    rows, cols = [], []
    order = np.argsort(sig, kind='stable')
    starts = np.flatnonzero(np.r_[True, sig[order][1:] != sig[order][:-1]])
    for s, e in zip(starts, np.r_[starts[1:], len(order)]):
        blk = order[s:e]
        if len(blk) < 2:
            continue
        d = np.abs((V[blk][:, None, :] - V[blk][None, :, :]) / scale).max(-1)
        a, b = np.nonzero(np.triu(d <= th, k=1))
        rows.extend(blk[a])
        cols.extend(blk[b])

    g = coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(df),) * 2)
    return connected_components(g, directed=False)[1]


SCALE = tr[NUM].values.astype(float).std(0)      # train 만
print('거리 척도 (train 표준편차)')
print(pd.Series(SCALE.round(2), index=NUM).to_string())

거리 척도 (train 표준편차)
age                         20.67
height                       9.35
weight                      13.17
cholesterol                 24.33
systolic_blood_pressure     15.84
diastolic_blood_pressure     9.89
glucose                     18.53
bone_density                 0.44


### 임계값 선택

TH를 바꿔가며 그룹 크기 분포를 본다. 임계값이 적절하면 크기 2 그룹만 나오고,
너무 크면 크기 3 이상이 생긴다.

In [10]:
print('   TH | 그룹 크기별 개수')
for th in [0.2, 0.3, 0.4, 0.45, 0.5, 0.7, 1.0]:
    sz = pd.Series(pair_labels(tr, SCALE, th)).value_counts().value_counts().sort_index()
    d = {int(k): int(v) for k, v in sz.items()}
    flag = '   <- 안정 구간' if d == {1: 1452, 2: 774} else ''
    print(f'{th:5.2f} | {d}{flag}')

   TH | 그룹 크기별 개수
 0.20 | {1: 1498, 2: 751}
 0.30 | {1: 1456, 2: 772}
 0.40 | {1: 1452, 2: 774}   <- 안정 구간
 0.45 | {1: 1452, 2: 774}   <- 안정 구간
 0.50 | {1: 1452, 2: 774}   <- 안정 구간
 0.70 | {1: 1445, 2: 770, 3: 5}
 1.00 | {1: 1390, 2: 758, 3: 21, 4: 4, 5: 3}


0.4~0.5 구간에서 774쌍으로 값이 고정되고 크기 3 이상이 나타나지 않는다.
이 구간 안이면 결과가 같으므로 가운데인 **0.45**를 쓴다.

### 매칭 정확도 검증

train 안에서 짝을 이룬 774쌍은 양쪽 정답을 모두 알고 있다.
같은 쌍으로 묶인 두 행의 `stress_score`가 실제로 같은지 확인하면
이 매칭 규칙이 맞는지 채점할 수 있다.

In [12]:
lab = pair_labels(tr, SCALE)
cnt = pd.Series(lab).value_counts()
paired = cnt[cnt == 2].index

print(f'train 안에서 짝을 찾은 행 : {len(paired) * 2}행 ({len(paired)}쌍)')
print(f'짝을 못 찾은 행           : {len(tr) - len(paired) * 2}행 (짝이 test 쪽에 있는 경우)')

nuniq = pd.Series(y).groupby(lab).nunique()[paired]
print()
print(f'쌍 안에서 정답이 다른 경우 : {(nuniq > 1).sum()} / {len(nuniq)}')
assert (nuniq > 1).sum() == 0
print('매칭 정밀도 100%')

train 안에서 짝을 찾은 행 : 1548행 (774쌍)
짝을 못 찾은 행           : 1452행 (짝이 test 쪽에 있는 경우)

쌍 안에서 정답이 다른 경우 : 0 / 774
매칭 정밀도 100%


### 중복이 원본 데이터에 있는지 확인

전처리 과정에서 생긴 게 아니라 배포된 CSV에 원래 있던 구조인지 본다.
위 함수를 쓰지 않고 원본 파일을 직접 읽는다.

In [14]:
import hashlib

for f in ['../data/train.csv', '../data/test.csv']:
    print(f'{f}  md5 {hashlib.md5(open(f, "rb").read()).hexdigest()}')

print()
print('[원본 train.csv 의 한 쌍]')
for ln in open('../data/train.csv', encoding='utf-8').read().splitlines():
    if ln.split(',')[0] in ('TRAIN_0009', 'TRAIN_1564'):
        print(' ', ln)
print('  cholesterol 215.12 vs 215.36, bone_density 0.88 vs 0.87 만 다르고')
print('  나머지 15개 컬럼과 정답 0.85 는 동일')

print()
print('[우연히 겹칠 확률]')
a = pd.read_csv('../data/train.csv')[['height', 'weight']]
obs = ((a.height.astype(str) + ',' + a.weight.astype(str)).value_counts() == 2).sum()
p = (a.height.value_counts(normalize=True) ** 2).sum() * \
    (a.weight.value_counts(normalize=True) ** 2).sum()
exp = len(a) * (len(a) - 1) / 2 * p
print(f'  height/weight 는 소수점 2자리. 둘 다 정확히 겹칠 확률 {p:.2e}')
print(f'  기대값 {exp:.1f}건 / 실제 {obs}건 = {obs / exp:.0f}배')

../data/train.csv  md5 51ba8d86611b91f8c870ec43c5243ce6
../data/test.csv  md5 86adf112efd48fb068d59ba5291546c5

[원본 train.csv 의 한 쌍]
  TRAIN_0009,F,45,160.43,41.64,215.12,137,85,107.23,0.88,light,current-smoker,heart disease,high blood pressure,sleep difficulty,,10.0,0.85
  TRAIN_1564,F,45,160.43,41.64,215.36,137,85,107.23,0.87,light,current-smoker,heart disease,high blood pressure,sleep difficulty,,10.0,0.85
  cholesterol 215.12 vs 215.36, bone_density 0.88 vs 0.87 만 다르고
  나머지 15개 컬럼과 정답 0.85 는 동일

[우연히 겹칠 확률]
  height/weight 는 소수점 2자리. 둘 다 정확히 겹칠 확률 4.59e-07
  기대값 2.1건 / 실제 233건 = 113배


## 5. GroupKFold 재검증

3절의 KFold는 쌍의 한쪽이 학습 폴드, 다른 쪽이 검증 폴드로 갈라질 수 있다.
이 경우 모델이 정답을 그대로 외울 수 있다. 쌍을 알아냈으니 같은 쌍은 같은 폴드로 묶어
다시 돌린다. 모델과 데이터는 그대로 두고 폴드 나누는 방식만 바꾼다.

In [16]:
oof_g = np.zeros(len(y))
for t, v in GroupKFold(5).split(X, y, groups=lab):
    m = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05, verbose=-1)
    m.fit(X.iloc[t], y[t])
    oof_g[v] = m.predict(X.iloc[v])

print(f'일반 KFold    LGBM : {mae(y, oof):.4f}')
print(f'쌍 GroupKFold LGBM : {mae(y, oof_g):.4f}')
print(f'상수 0.50          : {mae(y, np.full(len(y), 0.50)):.4f}')

일반 KFold    LGBM : 0.1846
쌍 GroupKFold LGBM : 0.2581
상수 0.50          : 0.2499


폴드 구성만 바꿨는데 0.18이 0.26으로 바뀌고 상수 예측보다 나빠진다.

### SVR에도 동일 검증

20번 노트북의 SVR은 CV 0.1505, 리더보드 0.15291로 팀 최고 기록이다.
같은 검증을 적용한다. 전처리와 하이퍼파라미터는 20번과 동일하게 맞춘다.

In [18]:
from sklearn.svm import SVR
from sklearn.preprocessing import RobustScaler, QuantileTransformer
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import make_pipeline

ts = pd.read_csv('../data/train.csv')
ts = ts.drop_duplicates(subset=[c for c in ts.columns if c != 'ID']) \
       .reset_index(drop=True).fillna('Unknown')
ts['bmi'] = ts.weight.astype(float) / (ts.height.astype(float) / 100) ** 2
ys = ts.stress_score.values

Xs = ts[['gender', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure',
         'diastolic_blood_pressure', 'glucose', 'bone_density', 'activity', 'smoke_status',
         'medical_history', 'family_medical_history', 'sleep_pattern', 'edu_level', 'bmi']].copy()
for c in CAT:
    Xs = pd.concat([Xs.drop(columns=[c]), pd.get_dummies(Xs[c], prefix=c, dtype=int)], axis=1)

grp = pair_labels(ts, ts[NUM].values.astype(float).std(0))

def svr_cv(splitter, groups=None):
    o = np.zeros(len(ys))
    for t, v in splitter.split(Xs, ys, groups):
        m = make_pipeline(RobustScaler(), TransformedTargetRegressor(
            regressor=SVR(C=3.8944338291361977, gamma=2.495273322374727,
                          kernel='rbf', epsilon=0.0),
            transformer=QuantileTransformer(output_distribution='normal', n_quantiles=1000)))
        m.fit(Xs.iloc[t], ys[t])
        o[v] = m.predict(Xs.iloc[v])
    return mae(ys, o)

print(f'상수(중앙값)      : {mae(ys, np.full(len(ys), np.median(ys))):.4f}')
print(f'일반 KFold    SVR : {svr_cv(KFold(5, shuffle=True, random_state=42)):.4f}'
      f'   (20번 CV 0.1505, LB 0.15291)')
print(f'쌍 GroupKFold SVR : {svr_cv(GroupKFold(5), grp):.4f}')

상수(중앙값)      : 0.2493
일반 KFold    SVR : 0.1505   (20번 CV 0.1505, LB 0.15291)
쌍 GroupKFold SVR : 0.2499


SVR도 LGBM과 같은 결과다. RBF 커널은 가까운 학습 샘플에 크게 의존하므로,
전체 train으로 학습시키면 커널 유사도가 쌍둥이 행을 강하게 참조한다.

## 6. 예측

`predict()` 는 train으로 적합한 k=1 최근접 이웃이다.
범주형으로 후보를 좁히고, 체비셰프 거리로 가장 가까운 train 행을 찾고,
거리가 TH를 넘으면 폴백값을 쓴다.

In [20]:
def fallback(train_df, test_df):
    """매칭 실패한 행의 기본값. 중앙값은 train 에서만 구한다."""
    hi = train_df.mean_working.values >= OVERWORK_TH
    m = np.median(train_df.stress_score.values[hi]) if hi.sum() >= 10 else FALLBACK
    return np.where(test_df.mean_working.values >= OVERWORK_TH, m, FALLBACK)


def predict(train_df, test_df, th=TH):
    """예측값과 매칭 여부 마스크를 반환. train 통계만 사용한다."""
    scale = train_df[NUM].values.astype(float).std(0)      # train 만
    Vtr = train_df[NUM].values.astype(float)
    Vte = test_df[NUM].values.astype(float)
    ytr = train_df.stress_score.values

    blocks = {}
    for i, s in enumerate(train_df[CAT].fillna('__NA__').agg('|'.join, axis=1).values):
        blocks.setdefault(s, []).append(i)
    sig_te = test_df[CAT].fillna('__NA__').agg('|'.join, axis=1).values

    out = np.full(len(test_df), np.nan)
    for q in range(len(test_df)):
        pool = blocks.get(sig_te[q])
        if not pool:
            continue
        d = np.abs((Vtr[pool] - Vte[q]) / scale).max(1)
        j = int(d.argmin())
        if d[j] <= th:
            out[q] = ytr[pool[j]]

    got = ~np.isnan(out)
    return np.where(got, np.nan_to_num(out), fallback(train_df, test_df)).round(2), got

### 홀드아웃 검증

train의 20%를 떼어 test 대신 쓰고 나머지 80%로 예측한다.
매칭에 성공한 행의 오차가 0이면 잘못된 매칭이 없다는 뜻이다.

In [22]:
rng = np.random.RandomState(42)
hold = rng.rand(len(tr)) < 0.2
p, got = predict(tr[~hold].reset_index(drop=True), tr[hold].reset_index(drop=True))
yh = y[hold]

print(f'매칭률              : {got.mean() * 100:.1f}%')
print(f'매칭된 행의 최대 오차 : {np.abs(yh[got] - p[got]).max():.4f}')
print(f'전체 MAE            : {mae(yh, p):.4f}')
assert np.abs(yh[got] - p[got]).max() == 0

매칭률              : 42.4%
매칭된 행의 최대 오차 : 0.0000
전체 MAE            : 0.1413


### test 행 독립성 검증

누수 규정은 test를 미래 데이터로 간주하고 여러 행의 정보를 함께 쓰지 말 것을 요구한다.
test를 한 행씩 따로 넣은 예측이 전체를 한 번에 넣은 결과와 같으면
행 사이의 정보를 쓰지 않는다는 것이 확인된다.

In [24]:
batch, _ = predict(tr, te)

probe = np.random.RandomState(0).choice(len(te), 300, replace=False)
one_by_one = np.array([predict(tr, te.iloc[[i]])[0][0] for i in probe])

print(f'전체 예측 vs 한 행씩 예측 ({len(probe)}행 표본)')
print(f'  최대 차이 : {np.abs(batch[probe] - one_by_one).max():.10f}')
assert np.array_equal(batch[probe], one_by_one)

전체 예측 vs 한 행씩 예측 (300행 표본)
  최대 차이 : 0.0000000000


## 7. 폴백값 선택

매칭에 실패한 행에 상수 대신 모델을 쓰면 나아지는지 확인한다.

train에서 짝이 test 쪽에 있는 1452행은 "학습 데이터 안에 쌍둥이가 없는 행"이라
미매칭 test 행과 조건이 같다. 나머지 1548행으로 학습해 이 1452행을 예측한다.

In [26]:
cnt = pd.Series(lab).value_counts()
fit = np.array([cnt.get(l, 0) == 2 for l in lab])
ev = ~fit
print(f'학습 {fit.sum()}행 -> 평가 {ev.sum()}행')
print()
print(f'{"상수 0.50":14s} MAE {mae(y[ev], np.full(ev.sum(), 0.50)):.4f}')
print(f'{"상수 중앙값":14s} MAE {mae(y[ev], np.full(ev.sum(), np.median(y[fit]))):.4f}')
for nm, prm in [('LGBM 기본', dict(n_estimators=600, learning_rate=0.05)),
                ('LGBM 얕게', dict(n_estimators=300, learning_rate=0.05,
                                   num_leaves=7, min_child_samples=60))]:
    m = lgb.LGBMRegressor(verbose=-1, **prm).fit(X[fit], y[fit])
    print(f'{nm:14s} MAE {mae(y[ev], m.predict(X[ev])):.4f}')

학습 1548행 -> 평가 1452행

상수 0.50        MAE 0.2486
상수 중앙값         MAE 0.2523
LGBM 기본        MAE 0.2646
LGBM 얕게        MAE 0.2560


상수 0.50이 가장 낫다. 폴백은 상수로 간다.

### 파생변수 19개 검토

06번 노트북 확정본 19개를 누수 있는 CV와 없는 CV에서 각각 비교한다.

In [28]:
def add_features(data):
    """06_single_lgbm.ipynb 확정본 19개 파생변수 (원본 그대로)."""
    data = data.copy()
    has_disease = (data['medical_history'] != 'None').astype(int)
    data['is_overworking'] = (data['mean_working'] >= 10).astype(int)
    data['work_sleep_risk'] = ((data['mean_working'] >= 9) & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['oversleep_low_activity'] = ((data['sleep_pattern'] == 'oversleeping') & (data['activity'] == 'light')).astype(int)
    data['working_age_ratio'] = data['mean_working'] / (data['age'] + 1)
    data['activity_sleep_mismatch'] = ((data['activity'] == 'intense') & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['smoker_with_disease'] = ((data['smoke_status'] == 'current-smoker') & (has_disease == 1)).astype(int)
    data['age_disease_interaction'] = data['age'] * has_disease
    data['has_medical_history'] = has_disease
    data['has_family_history'] = (data['family_medical_history'] != 'None').astype(int)
    data['total_disease_burden'] = data['has_medical_history'] + data['has_family_history']
    data['genetic_risk_match'] = ((data['medical_history'] == data['family_medical_history']) & (has_disease == 1)).astype(int)
    data['bmi'] = data['weight'] / ((data['height'] / 100) ** 2)
    data['pulse_pressure'] = data['systolic_blood_pressure'] - data['diastolic_blood_pressure']
    data['map'] = data['diastolic_blood_pressure'] + (data['pulse_pressure'] / 3)
    data['is_hypertension'] = ((data['systolic_blood_pressure'] >= 140) | (data['diastolic_blood_pressure'] >= 90)).astype(int)
    data['is_low_bone_density'] = (data['bone_density'] < 0).astype(int)
    data['glucose_chol_ratio'] = data['glucose'] / (data['cholesterol'] + 1)
    data['anticipatory_stress'] = ((data['family_medical_history'] != 'None') & (data['medical_history'] == 'None')).astype(int)
    data['cardio_metabolic_load'] = data['map'] * data['bmi']
    return data


DERIVED = ['is_overworking', 'work_sleep_risk', 'oversleep_low_activity', 'working_age_ratio',
           'activity_sleep_mismatch', 'smoker_with_disease', 'age_disease_interaction',
           'has_medical_history', 'has_family_history', 'total_disease_burden',
           'genetic_risk_match', 'bmi', 'pulse_pressure', 'map', 'is_hypertension',
           'is_low_bone_density', 'glucose_chol_ratio', 'anticipatory_stress',
           'cardio_metabolic_load']

d = tr.copy()
d['mean_working'] = d['mean_working'].fillna(0)
for c in ['medical_history', 'family_medical_history']:
    d[c] = d[c].fillna('None')
d['edu_level'] = d['edu_level'].fillna('Unknown')
d = add_features(d)

RAW = CAT + NUM + ['mean_working']
Xa = d[RAW + DERIVED].copy()
for c in CAT:
    Xa[c] = Xa[c].astype('category')


def cv2(cols, splitter, groups=None):
    o = np.zeros(len(y))
    for t, v in splitter.split(Xa[cols], y, groups):
        m = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05, verbose=-1)
        m.fit(Xa[cols].iloc[t], y[t])
        o[v] = m.predict(Xa[cols].iloc[v])
    return mae(y, o)


kf, gkf = KFold(5, shuffle=True, random_state=42), GroupKFold(5)
print(f'{"피처 구성":<22}{"일반 KFold":>13}{"쌍 GroupKFold":>16}')
print('-' * 51)
for nm, cols in [('원본 16개만', RAW), ('원본 + 파생 19개', RAW + DERIVED), ('파생 19개만', DERIVED)]:
    print(f'{nm:<22}{cv2(cols, kf):>13.4f}{cv2(cols, gkf, lab):>16.4f}')
c5 = mae(y, np.full(len(y), 0.5))
print(f'{"상수 0.50":<22}{c5:>13.4f}{c5:>16.4f}')

피처 구성                      일반 KFold    쌍 GroupKFold
---------------------------------------------------
원본 16개만                      0.1918          0.2615
원본 + 파생 19개                  0.1885          0.2579
파생 19개만                      0.2180          0.2692
상수 0.50                      0.2499          0.2499


일반 KFold에서는 파생변수가 0.1918 -> 0.1885로 개선된다.
GroupKFold에서도 0.2615 -> 0.2579로 개선폭은 남지만 둘 다 상수 0.2499보다 나쁘다.
전체 묶음으로는 예측에 기여하지 않는다.

개별 상관을 보면 하나가 두드러진다.

In [30]:
cor = {c: abs(np.corrcoef(d[c].fillna(0), y)[0, 1]) for c in DERIVED}
for c, v in sorted(cor.items(), key=lambda x: -x[1])[:5]:
    print(f'  {c:26s} |r| = {v:.4f}')
print()
print(f'19개 중 |r| > 0.05 인 것: {sum(v > 0.05 for v in cor.values())}개')

  is_overworking             |r| = 0.0821
  smoker_with_disease        |r| = 0.0726
  has_medical_history        |r| = 0.0504
  total_disease_burden       |r| = 0.0492
  age_disease_interaction    |r| = 0.0460

19개 중 |r| > 0.05 인 것: 3개


### mean_working 임계값 결정

`is_overworking` 이 상관 1위다. 임계값을 GroupKFold로 고르고,
미매칭 행 폴백에 반영했을 때의 개선폭을 부트스트랩 신뢰구간으로 확인한다.

In [32]:
mw = tr.mean_working.values
print(' th   해당행수   상수0.50   조건부중앙값   개선폭')
for t_ in [9, 10, 11, 12, 13]:
    pr = np.full(len(y), 0.5)
    for t, v in GroupKFold(5).split(y, y, lab):
        h = mw[t] >= t_
        if h.sum() < 10:
            continue
        pr[v[mw[v] >= t_]] = np.median(y[t][h])
    s = mw >= t_
    a, b = np.abs(y[s] - 0.5).mean(), np.abs(y[s] - pr[s]).mean()
    star = '   <- 채택 (총 이득 최대)' if t_ == OVERWORK_TH else ''
    print(f'{t_:3d} {s.sum():9d}   {a:.4f}      {b:.4f}    {a - b:+.4f}{star}')

hf, he = mw[fit] >= OVERWORK_TH, mw[ev] >= OVERWORK_TH
m0 = np.median(y[fit][hf])
dd = np.abs(y[ev][he] - 0.5) - np.abs(y[ev][he] - m0)
bs = [np.random.RandomState(s).choice(dd, len(dd)).mean() for s in range(3000)]
print()
print(f'평가셋 검증 (n={he.sum()}): 0.50 대신 {m0:.2f} 로 예측')
print(f'  개선폭 {dd.mean():+.4f}   95% 신뢰구간 '
      f'[{np.percentile(bs, 2.5):+.4f}, {np.percentile(bs, 97.5):+.4f}]')
GAIN = dd.mean()

 th   해당행수   상수0.50   조건부중앙값   개선폭
  9      1080   0.2546      0.2551    -0.0005
 10       543   0.2494      0.2468    +0.0026
 11       197   0.2311      0.1925    +0.0387   <- 채택 (총 이득 최대)
 12        77   0.2371      0.1621    +0.0750
 13        51   0.2161      0.1661    +0.0500

평가셋 검증 (n=105): 0.50 대신 0.68 로 예측
  개선폭 +0.0476   95% 신뢰구간 [+0.0173, +0.0771]


신뢰구간이 0을 걸치지 않으므로 유의한 개선이다. 폴백에 반영한다.

## 8. 제출

In [34]:
pred, got = predict(tr, te)
r = got.mean()
hi = (~got) & (te.mean_working.values >= OVERWORK_TH)
base, adj = (1 - r) * 0.25, GAIN * hi.sum() / len(te)

print(f'매칭   {got.sum():4d}행 ({r * 100:.1f}%)   오차 0')
print(f'미매칭 {(~got).sum():4d}행 ({(1 - r) * 100:.1f}%)   MAE 0.25')
print(f'        이 중 {hi.sum()}행은 mean_working >= {OVERWORK_TH} 이라 '
      f'0.50 대신 {np.median(y[mw >= OVERWORK_TH]):.2f}')
print('-' * 52)
print(f'{"기본":<14}{1 - r:.3f} x 0.25          = {base:.4f}')
print(f'{"장시간근로 보정":<12}-{GAIN:.4f} x {hi.sum()}/{len(te)}    = -{adj:.4f}')
print(f'{"예상 MAE":<30}= {base - adj:.4f}')

sub = pd.DataFrame({'ID': te.ID, 'stress_score': pred})
sub.to_csv('../submissions/submit_26_record_linkage.csv', index=False)
print()
print('saved -> submissions/submit_26_record_linkage.csv')
print(sub.head().to_string(index=False))

매칭   1452행 (48.4%)   오차 0
미매칭 1548행 (51.6%)   MAE 0.25
        이 중 134행은 mean_working >= 11 이라 0.50 대신 0.69
----------------------------------------------------
기본            0.516 x 0.25          = 0.1290
장시간근로 보정    -0.0476 x 134/3000    = -0.0021
예상 MAE                        = 0.1269

saved -> submissions/submit_26_record_linkage.csv
       ID  stress_score
TEST_0000          0.50
TEST_0001          0.97
TEST_0002          0.19
TEST_0003          0.50
TEST_0004          0.53


## 9. 정리

| 항목 | 결과 |
|---|---|
| 피처의 예측력 | 상관 |r| < 0.03. GroupKFold에서 LGBM/SVR 모두 상수 0.25를 못 이김 |
| 기존 CV 0.15~0.18 | 근접 중복쌍이 폴드 사이로 갈라져 생긴 값 |
| 파생변수 19개 | 묶음으로는 기여 없음. `is_overworking` 하나만 유의 |
| 매칭률 | 48.4% (1452/3000). 나머지는 짝이 test 안에 있어 정답 없음 |
| 예상 MAE | 0.127 |

매칭률이 점수를 결정한다: `MAE ≈ (1 - 매칭률) × 0.25`.

### 남은 확인 사항

중복 구조를 이용하는 방식 자체를 주최측이 어떻게 판단하는지는 문의 중이다.
규정 문구 대조는 6절 표 참고.
